In [1]:
import numpy as np
from pymoo.indicators.hv import HV
from bikeroot import BikeNetwork, CostApplicator, Cost, NgaMobrp_Cost, alg_utils, eval_utils
import pickle
import os
from bikeroot import Result, biobjective_label_setting_solve, biobjective_label_correcting_solve, hmls_solve, ngamobrp_solve, nga3mobrp_solve, moedbrp_solve
import time


locations = ['Spokane, WA', "San Francisco, CA", "Prague", "Aveiro"]
path_distances = [500, 1500, 3000, 5000, 10000, 15000, 20000] #by meters

DOWNLOAD_MAPS = True
GENERATE_CANDIDATES = False
RUN_TEST_1 = False
TEST_REF_POINTS = False

In [3]:

if DOWNLOAD_MAPS:
    # completed = pickle.load(open('completed_locations.pkl', 'rb')) if 'completed_locations.pkl' in os.listdir() else []
    for location in locations:
        # if location in completed:
        #     print(f"Skipping {location}, already processed.")
        #     continue
        print(f"Processing {location}...")
        bike_net = BikeNetwork(location)
        size = 20000
        # size = max(path_distances) + 5000
        loc_name = location.split(",")[0].lower().replace(" ", "_")
        bike_net.load_graph(f'data/testing_maps/{loc_name}_{size}.graphml')
        cost = NgaMobrp_Cost()
        g_with_cost = CostApplicator.apply_cost_functions(bike_net,cost)
        bike_net.update_graph(g_with_cost)
        bike_net.save_graph(f'data/testing_maps_2/{loc_name}_{size}_withcost.graphml')
        # completed.append(location)
        # pickle.dump(completed, open('completed_locations.pkl', 'wb'))
    

Processing Spokane, WA...
Processing San Francisco, CA...
Processing Prague...
Processing Aveiro...


In [ ]:
if GENERATE_CANDIDATES:
    for location in locations:
        print(f"Generating candidates for {location}...")
        loc_name = location.split(",")[0].lower().replace(" ", "_")
        candidate_file_path = f'data/testing_candidates_2/{loc_name}_candidates.pkl'
        candidates = pickle.load(open(candidate_file_path, 'rb')) if os.path.isfile(candidate_file_path) else {}
        print(f"Loaded {len(candidates)} candidates from {candidate_file_path}")

        if candidates == {}:
            for path_distance in path_distances:
                candidates[path_distance] = []
        
        if all([len(candidates[dist]) > 5 for dist in path_distances]):
            print(f"All candidates for {location} already generated.")
            continue
        bike_net = BikeNetwork(location)
        bike_net.load_graph(f'data/testing_maps/{loc_name}_20000_withcost.graphml')



        while not all([len(candidates[dist]) > 5 for dist in path_distances]):
            nodes = bike_net.random_nodes(2)
            src, dst = str(nodes[0]), str(nodes[1])
            # print(start, end)
            if src == dst:
                continue
            src_coords = bike_net.graph.nodes[src]['y'], bike_net.graph.nodes[src]['x']
            dst_coords = bike_net.graph.nodes[dst]['y'], bike_net.graph.nodes[dst]['x']

            path_cost = alg_utils.haversine_distance(src_coords, dst_coords)
            
            for path_distance in path_distances:
                if abs(path_cost - path_distance) < 150:
                    candidates[path_distance].append({'src': src, 'dst': dst, 'cost': path_cost})
                    # print(f"Updated candidate for {path_distance}m: {src} to {dst} with cost {path_cost}")
        pickle.dump(candidates, open(candidate_file_path, 'wb'))
            

    # for loc in locations:
    #     loc_name = loc.split(",")[0].lower().replace(" ", "_")
    #     candidates = pickle.load(open(f'{loc_name}_candidates.pkl', 'rb'))
    #     all_candidates = pickle.load(open(f'{loc_name}_allcandidates.pkl', 'rb'))
    #     print(f"Location: {loc}")
    #     print(f"Candidates: {candidates}")
    #     print(f"All Candidates: {len(all_candidates)} paths found")

Generating candidates for Spokane, WA...
Loaded 7 candidates from data/testing_candidates_2/spokane_candidates.pkl
All candidates for Spokane, WA already generated.
Generating candidates for San Francisco, CA...
Loaded 7 candidates from data/testing_candidates_2/san_francisco_candidates.pkl
Generating candidates for Prague...
Loaded 0 candidates from data/testing_candidates_2/prague_candidates.pkl
Generating candidates for Aveiro...
Loaded 0 candidates from data/testing_candidates_2/aveiro_candidates.pkl


In [7]:
if RUN_TEST_1:
    location = "Spokane, WA"    
    loc_name = location.split(",")[0].lower().replace(" ", "_")
    candidates = pickle.load(open(f'data/testing_candidates/{loc_name}_candidates.pkl', 'rb'))
    print(candidates)

    bike_net = BikeNetwork(location)
    bike_net.load_graph(f'data/testing_maps/{loc_name}_20000_withcost.graphml')
    distance = 500
    src = str(candidates[distance]['src'])
    dst = str(candidates[distance]['dst'])
    criteria = ['travel distance','travel time', 'safety', 'comfort']

    bilabelsetting = {'func': biobjective_label_setting_solve, 'params':{}, 'name': 'biobjective_label_setting_solve'}

    bilabelcorrecting = {'func': biobjective_label_correcting_solve, 'params':{}, 'name': 'biobjective_label_correcting_solve'}

    heuristiclabelset = {
        'func':heuristic_enabled_mc_label_setting_solve,
        'params':{
            'heuristic_skip': HeuristicSkip(type='ellipse',G=bike_net.graph, src=src, dst=dst, ab=3.5),
            'heuristic_prune': None,#HeuristicPruning(type='ratio', dst=dst, G=bike_net.graph, criteria='travel distance', alpha=2),
            'heuristic_checkdom': HeuristicCheckDom(type='epsilon', G=bike_net.graph, criteria=criteria, epsilon=0.00)
            },
        'name': 'heuristic_enabled_mc_label_setting_solve'
        }

    ngamobrpset = {'func':ngamobrp_solve, 'params':{'debug':False}, 'name': 'ngamobrp_solve'}
    nga3mobrpset = {'func':nga3mobrp_solve, 'params':{'debug':False}, 'name': 'nga3mobrp_solve'}
    moedbrpset = {'func':moedbrp_solve, 'params':{'debug':False}, 'name': 'moedbrp_solve'}


    algs = [ngamobrpset]
    results = pickle.load(open('data/testing/results.pkl', 'rb')) if os.path.isfile('data/testing/results.pkl') else []
    print(f"Loaded {len(results)} previous results.")
    print(f"Running tests for {location} from {src} to {dst} with distance {distance}m and criteria {criteria}")
    bike_net.weights_to_floats(criteria)

    for alg in algs:
        print(f"Running algorithm: {alg['name']}")
        start_time = time.time()
        path_out = alg['func'](G=bike_net.graph, src=src, dst=dst, criteria=criteria, **alg['params'])
        end_time = time.time()
        result = Result(
            graph_name = f'data/testing_maps/{loc_name}_20000_withcost.graphml',
            algorithm=alg['name'],
            location=location,
            distance=distance,
            src=src,
            dst=dst,
            criteria=criteria,
            params=alg['params']
        )
        print(path_out, f"Time taken: {end_time - start_time:.2f} seconds")
        print(len(path_out))
        result.add_result(path_out,None,end_time - start_time)
        results.append(result)
        pickle.dump(results, open('data/testing/results.pkl', 'wb'))


In [8]:
# results = pickle.load(open('data/testing/results.pkl', 'rb'))
# for result in results:
#     print(result)
#     print(result.times)

In [9]:
if TEST_REF_POINTS:
    location = "Spokane, WA"    
    loc_name = location.split(",")[0].lower().replace(" ", "_")
    candidates = pickle.load(open(f'data/testing_candidates/{loc_name}_candidates.pkl', 'rb'))
    print(candidates)

    bike_net = BikeNetwork(location)
    bike_net.load_graph(f'data/testing_maps/{loc_name}_20000_withcost.graphml')
    criteria = ['travel distance','travel time', 'safety', 'comfort']
    
    results = pickle.load(open('data/testing/results.pkl', 'rb')) if os.path.isfile('data/testing/results.pkl') else []
    distance = 500
    src = str(candidates[distance]['start'])
    dst = str(candidates[distance]['end'])
    ref_points = eval_utils.calculate_refpoint(bike_net.graph, src, dst, criteria)
    print(f"Reference points for {src} to {dst}: {ref_points}")

In [10]:
# for loc in locations:
#     loc_name = loc.split(",")[0].lower().replace(" ", "_")
#     candidates = pickle.load(open(f'data/testing_candidates/{loc_name}_candidates.pkl', 'rb'))
#     candidates_renamed = {}
#     changed =  False
#     for dist, data in candidates.items():
#         if data['cost'] is not None and 'src' not in data.keys():
#             changed = True
#             candidates_renamed[dist] = {
#                 'cost': data['cost'],
#                 'src': data['start'],
#                 'dst': data['end']
#             }
#     if changed:
#         pickle.dump(candidates_renamed, open(f'data/testing_candidates/{loc_name}_candidates.pkl', 'wb'))
